## LangSmith Prompt Hub
> LangSmith Prompt Hub -> 프롬프트 중앙 관리 시스템

### .env 파일에 API 키 저장

In [ ]:
from dotenv import load_dotenv

# 환경 변수 로드
load_dotenv()

In [ ]:
import os

# API 키 확인
api_key = os.getenv("LANGSMITH_API_KEY")
if api_key:
    print("LangSmith API 키가 설정되었습니다.")
else:
    print("LangSmith API 키가 없습니다. .env 파일을 확인해주세요.")

### LangSmith Client 생성

In [ ]:
from langsmith import Client

# LangSmith 클라이언트 생성
client = Client()

## Push Prompt (프롬프트 업로드)
### 기본 프롬프트 업로드

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# 프롬프트 템플릿 생성
prompt = ChatPromptTemplate.from_template(
    template="당신은 {topic}에 대한 전문가입니다. {topic}에 대해 간단하게 설명해주세요."
)

In [ ]:
# 프롬프트에 들어갈 변수
prompt.input_variables

In [ ]:
# LangSmith에 프롬프트 업로드
url = client.push_prompt(
    prompt_identifier="topic-explainer", # 프롬프트 이름
    object=prompt,                       # 업로드할 프롬프트 객체
    is_public=False                      # 공개 유무
)

print(f"프롬프트가 업로드되었습니다!: {url}")

### 모델과 함께 프롬프트 업로드
- 프롬프트와 모델 설정을 함께 저장할 수 있음
- 프롬프트를 불러올 때 모델 설정도 함께 가져올 수 있음

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

# 모델 생성
model = ChatGroq(
    model="openai/gpt-oss-120b",    # 모델 명
    temperature=0.1                 # 값이 낮을 수록 안정적+정확한 답변이 옴
)

In [ ]:
# 프롬프트 생성
prompt = ChatPromptTemplate.format_prompt(
    "{topic}에 대한 재미있는 농담을 하나 들려주세요."
)

In [ ]:
# 프롬프트와 모델을 체인으로 연결
chain = prompt | model

In [ ]:
# LangSmith에 업로드
url = client.push_prompt(
    prompt_identifier="joke-generator-with-model",  # 프롬프트 이름
    object=chain,
    is_public=False # 공개 유무
)

print(f"프롬프트와 모델이 함께 업로드되었습니다: {url}")

## Pull Prompt(프롬프트 가져오기)
### 기본 프롬프트 가져오기

In [ ]:
# 프롬프트 가져오기
# topic-explainer이름의 프롬프트를 가져오기
pulled_prompt = client.pull_prompt("topic-explainer")

# 가져온 프롬프트를 출력
print("프롬프트를 가져왔습니다!: ")
print(pulled_prompt)

In [ ]:
# 가져온 프롬프트에 넣어야하는 변수 확인
# topic-explainer이 프롬프트
pulled_prompt.input_variables

### 가져온 프롬프트 사용하기(topic-explainer)

In [ ]:
from langchain_ollama import ChatOllama

# 로컬 모델 생성
# topic-explainer는 모델이 아직 안붙고 단순 프롬프트
model = ChatOllama(
    model="gemma4:e4b",     # 모델 명
    temperature = 0.1       # 값이 작으면 정확+안정적인 대답이 나옴
)

In [ ]:
# chain 형성
# 가져온 프롬프트 | 로컬에서 형성한 모델
chain = pulled_prompt | model

In [ ]:
# 체인 실행
# pulled_prompt에 topic을 입력(양자컴퓨팅)
# .invoke() : 모델에 프롬프트를 입력해서 대답을 가져옴
response = chain.invoke({"topic":"양자컴퓨팅"})

# 모델에서 받아온 대답만 출력
print(response.content)

NameError: name 'chain' is not defined

### 모델과 함께 가져오기

In [ ]:
# 프롬프트와 모델을 함께 가져오기
# 모델과 같이 형성했던 joke-generator-with-model을 불러옴
# 해당 프롬프트는 모델이 있기 때문에 바로 chain으로 연결
chain = client.pull_prompt(
    prompt_identifier="joke-generator-with-model",
    include = True          # 프롬프트와 모델을 같이 가져올 수 있는 설정
)

In [ ]:
# 설정한 chain 값 불러오기
chain

In [ ]:
# 모델과 프롬프트가 같이 있기 때문에 topic을 입력하면 바로 사용가능
response = chain.invoke({"topic":"프로그래머"})

# 모델에서 받아온 대답만 출력
print(response.content)

NameError: name 'chain' is not defined

### 공개 프롬프트를 가져오기
> LangChain Hub의 공개 프롬프트를 가져올 수 있음

In [ ]:
# 공개 프롬프트 가져오기 (작성자/프롬프트명)
public_prompt = client.pull_prompt("heun0420/ko-summary")

# 공개 프롬프트에 대한 내용
public_prompt

In [ ]:
# 공개 프롬프트에 들어가야하는 변수
public_prompt.input_variables

In [ ]:
print(public_prompt.messages[0].prompt.template)
# 작성된 프롬프트가 어떤 형식인지에 대해서 출력

## List Prompts(프롬프트 목록 조회)
### 모든 프롬프트 조회

In [ ]:
# 내 워크페이스의 모든 프롬프트 조회
prompts = client.list_prompts()

In [ ]:
print("=== 프롬프트 목록 ===")
# Langsmith에 저장된 프롬프트 저장소 리스트 순회
# .repos : 프롬프트 저장소 리스트
for prompt in prompts.repos:
    # .repo_handle : 각 프롬프트의 이름 or 식별자 
    print(f"- {prompt.repo_handle}")

### 검색 조건으로 프롬프트 조회

In [ ]:
# 특정 키워드를 포함한 비공개 프롬프트만 조회
filltered_prompts = client.list_prompts(
    query="joke",       # 검색 키워드
    is_public=False     # 비공개 프롬프트만
)

In [ ]:
print("=== 'joke'를 포함한 비공개 프롬프트 ===")
# Langsmith에 저장된 프롬프트 저장소 리스트 순회
# .repos : 프롬프트 저장소 리스트
for prompt in prompts.repos:
    # 그중 joke를 포함한 비공개 프롬프트만 출력
    print(f"- {prompt.repo_handle}")

## Delete Prompt(프롬프트 삭제)
### topic-explainer
- 삭제하기전 프롬프트 확인

In [ ]:
# topic-explainer이런 이름에 비공개인 템플렛을 출력
client.list_prompts(
    query="topic-explainer",    # 검색 키워드
    is_public=Fasle             # 비공개 프롬프트만
)

In [ ]:
# 프롬프트 삭제(주의: 실제로 삭제됩니다!)
client.delete_prompt("topic-explainer")

In [ ]:
# 프롬프트를 삭제하였기 때문에 해당 프롬프트는 출력되지 않음
client.list_prompts(
    query="topic-explainer",    # 검색 키워드
    is_public=Fasle             # 비공개 프롬프트만
)

### joke-generator-with-model
- 삭제하기 전 프롬프트 확인

In [ ]:
# joke-generator-with-model이런 이름에 비공개인 템플렛을 출력
client.list_prompts(
    query="joke-generator-with-model",  # 검색 키워드
    is_public=Fasle                     # 비공개 프롬프트만
)

In [ ]:
# 프롬프트 삭제(주의: 실제로 삭제됩니다!)
client.delete_prompt("joke-generator-with-model")

In [ ]:
# 프롬프트를 삭제하였기 때문에 해당 프롬프트는 출력되지 않음
client.list_prompts(
    query="joke-generator-with-model",  # 검색 키워드
    is_public=Fasle                     # 비공개 프롬프트만
)